# P3-1 · Tokenization và embedding

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

In [ ]:
!pip install -q tiktoken

## Lý thuyết

In [ ]:
# Cần mạng: tải tokenizer từ Hugging Face (chạy trên Colab)
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
ids = tok("Tôi đang học trí tuệ nhân tạo.")["input_ids"]
len(ids), tok.convert_ids_to_tokens(ids)

In [ ]:
import torch
emb = torch.nn.Embedding(num_embeddings=32_000, embedding_dim=512)
ids = torch.tensor([[51, 4410, 15592]])     # (batch=1, seq=3)
x = emb(ids)                                # (1, 3, 512)
x.shape

torch.Size([1, 3, 512])


## Bài tập 4: So token tiếng Việt và tiếng Anh {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
# Cần mạng: tiktoken tải bảng mã o200k_base lần đầu (chạy trên Colab hoặc máy bạn)
import tiktoken
enc = tiktoken.get_encoding("o200k_base")
vi = "Học máy giúp máy tính tự học từ dữ liệu mà không cần được lập trình cụ thể cho từng nhiệm vụ."
en = "Machine learning lets computers learn from data without being explicitly programmed for each task."
nv, ne = len(enc.encode(vi)), len(enc.encode(en))
print(f"tiếng Việt {nv} token | tiếng Anh {ne} token | tỉ lệ {nv / ne:.2f}")
print([enc.decode([t]) for t in enc.encode(vi)])

## Bài tập 5: Byte, ký tự và dấu tiếng Việt {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import unicodedata
vi = "Hôm nay trời đẹp, chúng tôi đi dạo ở hồ Hoàn Kiếm."
en = "The weather is nice today, we walk around Hoan Kiem lake."
ten_nfc = unicodedata.normalize("NFC", "Nguyễn")
ten_nfd = unicodedata.normalize("NFD", "Nguyễn")      # như văn bản copy từ một số trình soạn thảo, macOS
# TODO: byte_vi, byte_en (số byte UTF-8), len của hai dạng tên, bằng nhau?

In [ ]:
for ten, s in [("vi", vi), ("en", en)]:
    print(f"{ten}: {len(s)} ký tự, {len(s.encode('utf-8'))} byte")
print(len(ten_nfc), len(ten_nfd), "bằng nhau?", ten_nfc == ten_nfd)
print("sau khi chuẩn hóa NFC:", unicodedata.normalize("NFC", ten_nfd) == ten_nfc)

vi: 50 ký tự, 68 byte
en: 57 ký tự, 57 byte
6 8 bằng nhau? False
sau khi chuẩn hóa NFC: True


## Bài tập 6: Áp dụng các phép gộp BPE lên từ mới {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
merges = [("n", "g"), ("ng", "u"), ("ê", "n"), ("y", "ễ"), ("yễ", "n"), ("ngu", "yễn"), ("i", "ê"), ("iê", "ng"), ("ng", "h")]

def encode(word, merges):
    tokens = list(word)
    # TODO: với mỗi (a, b) theo thứ tự, gộp mọi cặp liền nhau a, b thành a + b
    return tokens

In [ ]:
def encode(word, merges):
    tokens = list(word)
    for a, b in merges:
        i, out = 0, []
        while i < len(tokens):
            if i + 1 < len(tokens) and tokens[i] == a and tokens[i + 1] == b:
                out.append(a + b); i += 2
            else:
                out.append(tokens[i]); i += 1
        tokens = out
    return tokens

for w in ["nguyễn", "nguyên", "nghiêng"]:
    print(w, "→", encode(w, merges))

nguyễn → ['nguyễn']
nguyên → ['ngu', 'y', 'ên']
nghiêng → ['ngh', 'iêng']


## Bài tập 7: Tìm từ gần nghĩa bằng embedding {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import numpy as np
tu = ["vua", "nữ hoàng", "đàn ông", "phụ nữ", "mèo", "chó", "phở", "bánh mì"]
E = np.array([[0.9, 0.8, 0.0, 0.0], [0.9, -0.8, 0.0, 0.0], [0.1, 0.9, 0.1, 0.0], [0.1, -0.9, 0.1, 0.0],
              [0.0, 0.1, 0.9, 0.1], [0.0, 0.2, 0.8, 0.0], [0.0, 0.0, 0.1, 0.9], [0.0, 0.0, 0.0, 0.8]])

def gan_nhat(v, k=2, bo=()):
    # TODO: cosine của v với mọi hàng của E, bỏ các từ trong `bo`, trả về k từ cao nhất
    return []

In [ ]:
def gan_nhat(v, k=2, bo=()):
    cos = E @ v / (np.linalg.norm(E, axis=1) * np.linalg.norm(v))
    return [tu[i] for i in np.argsort(-cos) if tu[i] not in bo][:k]

meo = E[tu.index("mèo")]
cos_meo_cho = meo @ E[5] / (np.linalg.norm(meo) * np.linalg.norm(E[5]))
print("gần 'mèo':", gan_nhat(meo, 2, bo=("mèo",)))
v = E[tu.index("vua")] - E[tu.index("đàn ông")] + E[tu.index("phụ nữ")]
dap_an = gan_nhat(v, 1, bo=("vua", "đàn ông", "phụ nữ"))[0]
print("vua - đàn ông + phụ nữ ≈", dap_an)

gần 'mèo': ['chó', 'phở']
vua - đàn ông + phụ nữ ≈ nữ hoàng


## Bài tập 8: Embedding là phép tra bảng {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import torch, torch.nn as nn
torch.manual_seed(0)
emb = nn.Embedding(1000, 16)
ids = torch.tensor([5, 42, 5])
# TODO: a = emb(ids); b = one_hot @ emb.weight; giong; backward; so_dong

In [ ]:
a = emb(ids)
b = nn.functional.one_hot(ids, 1000).float() @ emb.weight
giong = torch.allclose(a, b)
a.pow(2).sum().backward()
so_dong = (emb.weight.grad.abs().sum(1) > 0).sum()
print(a.shape, "giống one-hot @ W:", giong, "| dòng có gradient:", so_dong.item())

torch.Size([3, 16]) giống one-hot @ W: True | dòng có gradient: 2


## Bài tập 9: RNN quên đầu câu thế nào {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import torch, torch.nn as nn
torch.manual_seed(0)
rnn = nn.RNN(input_size=8, hidden_size=32, batch_first=True)

def grad_dau(T):
    x = torch.randn(1, T, 8, requires_grad=True)
    out, h = rnn(x)
    h.sum().backward()
    return x.grad[0, 0].norm().item()
# TODO: g = {T: grad_dau(T) for T in (5, 20, 50)}

In [ ]:
g = {}
for T in (5, 20, 50):
    torch.manual_seed(1)
    g[T] = grad_dau(T)
    print(f"chuỗi {T:2d} bước: |∂h_cuối / ∂x_đầu| = {g[T]:.2e}")

chuỗi  5 bước: |∂h_cuối / ∂x_đầu| = 2.09e-01
chuỗi 20 bước: |∂h_cuối / ∂x_đầu| = 7.93e-06
chuỗi 50 bước: |∂h_cuối / ∂x_đầu| = 3.16e-14
